In [139]:
from google.cloud import storage
import os
from dotenv import load_dotenv
import pandas as pd
from io import BytesIO

load_dotenv()

True

In [140]:
GCS_BUCKET = os.getenv("GCS_BUCKET", "").strip()
GCS_MEDAL_FETCH = os.getenv("GCS_MEDAL_FETCH", "").strip()
BLOB_FETCH_PATH = f"{GCS_MEDAL_FETCH}/season=2026/laps/all_session_laps.parquet"

BLOB_FETCH_SESSIONS = f"gold/season=2026/session.parquet"
BLOB_FETCH_DRIVERS = f"gold/season=2026/driver.parquet"

In [141]:
client = storage.Client()
bucket = client.bucket(GCS_BUCKET)
blob = bucket.blob(BLOB_FETCH_PATH)
df = pd.read_parquet(BytesIO(blob.download_as_bytes()))

blob_sessions = bucket.blob(BLOB_FETCH_SESSIONS)
df_sessions = pd.read_parquet(BytesIO(blob_sessions.download_as_bytes()))

blob_drivers = bucket.blob(BLOB_FETCH_DRIVERS)
df_drivers = pd.read_parquet(BytesIO(blob_drivers.download_as_bytes()))

In [142]:
unused_columns = ["segments_sector_1", "segments_sector_2", "segments_sector_3"]
df = df.drop(columns=unused_columns)

In [143]:
used_sessions = df_sessions.loc[(df_sessions["session_name"] == "Sprint") | (df_sessions["session_name"] == "Race") & (df_sessions["is_cancelled"] == False)]

In [144]:
df = df.loc[df["session_key"].isin(used_sessions["session_key"])]

In [145]:
df_drivers = df_drivers[["driver_number", "full_name", "team_name", "country"]]

In [146]:
df_sessions = df_sessions[["session_key", "session_name", "location"]]

In [147]:
df = pd.merge(df, df_drivers, on="driver_number", how="left")
df = pd.merge(df, df_sessions, on="session_key", how="left")

In [148]:
df = df[[
    "meeting_key", 
    "session_key",
    "session_name",
    "location",
    "driver_number",
    "full_name",
    "team_name",
    "country",
    "lap_number",
    "date_start",
    "duration_sector_1",
    "duration_sector_2",
    "duration_sector_3",
    "i1_speed",
    "i2_speed",
    "is_pit_out_lap",
    "lap_duration",
    "st_speed",
    "ingested_at"
]]

In [149]:
df = df.sort_values(
    by=["session_key", "lap_number"],
    ascending=[True, False]
)

df = df.drop_duplicates(subset=["session_key", "driver_number"])

In [150]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 129 entries, 985 to 3895
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   meeting_key        129 non-null    int64              
 1   session_key        129 non-null    int64              
 2   session_name       129 non-null    object             
 3   location           129 non-null    object             
 4   driver_number      129 non-null    int64              
 5   full_name          129 non-null    object             
 6   team_name          129 non-null    object             
 7   country            129 non-null    object             
 8   lap_number         129 non-null    int64              
 9   date_start         129 non-null    datetime64[us, UTC]
 10  duration_sector_1  98 non-null     float64            
 11  duration_sector_2  96 non-null     float64            
 12  duration_sector_3  92 non-null     float64          

In [152]:
df["lap_duration"].isna().sum()

np.int64(37)

In [151]:
df

,meeting_key,session_key,session_name,location,driver_number,full_name,team_name,country,lap_number,date_start,duration_sector_1,duration_sector_2,duration_sector_3,i1_speed,i2_speed,is_pit_out_lap,lap_duration,st_speed,ingested_at
985,1279,11234,Race,Melbourne,63,George RUSSELL,Mercedes,United Kingdom,58,2026-03-08 05:26:33.622000+00:00,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,2026-05-04 18:23:14.237848+00:00
986,1279,11234,Race,Melbourne,12,Kimi ANTONELLI,Mercedes,Italy,58,2026-03-08 05:26:36.654000+00:00,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,2026-05-04 18:23:14.237848+00:00
992,1279,11234,Race,Melbourne,16,Charles LECLERC,Ferrari,Monaco,58,2026-03-08 05:26:49.137000+00:00,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,2026-05-04 18:23:14.237848+00:00
993,1279,11234,Race,Melbourne,44,Lewis HAMILTON,Ferrari,United Kingdom,58,2026-03-08 05:26:49.754000+00:00,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,2026-05-04 18:23:14.237848+00:00
997,1279,11234,Race,Melbourne,1,Lando NORRIS,McLaren,United Kingdom,58,2026-03-08 05:27:25.353000+00:00,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,2026-05-04 18:23:14.237848+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4827,1284,11280,Race,Miami Gardens,77,Valtteri BOTTAS,Cadillac,Finland,55,2026-05-03 18:36:50.714000+00:00,36.528,36.971,26.485,199.0,179.0,False,99.984,308.0,2026-05-04 18:23:14.237848+00:00
3960,1284,11280,Race,Miami Gardens,27,Nico HULKENBERG,Audi,Germany,8,2026-05-03 17:18:02.832000+00:00,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,2026-05-04 18:23:14.237848+00:00
3941,1284,11280,Race,Miami Gardens,30,Liam LAWSON,Racing Bulls,New Zealand,7,2026-05-03 17:15:18.820000+00:00,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,2026-05-04 18:23:14.237848+00:00
3890,1284,11280,Race,Miami Gardens,10,Pierre GASLY,Alpine,France,5,2026-05-03 17:10:33.708000+00:00,34.173,35.868,NaN,206.0,182.0,False,NaN,319.0,2026-05-04 18:23:14.237848+00:00
